In [5]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit.DataStructs import FingerprintSimilarity
from rdkit.Chem import PandasTools

In [6]:
df_csd = pd.read_csv('../data/csd_ligands_gd.csv')
df_csd

,Unnamed: 0,name,smiles,csd_entries
0,0,csd_000,CO[P@@](=O)(C(C)=[N:1]CC[N:1]1CC[N:1](CC[N:1]=...,ABAREB
1,1,csd_001,COc1ccc(C([P@](=O)(OC)[OH:1])=[N:1]CC[N:1]2CC[...,ABARIF
2,2,csd_002,Clc1ccc([OH:1])c(C=[N:1]CC[N:1](CC[N:1]=Cc2cc(...,ADUPEU
3,3,csd_003,Cn1cc[n:1]c1C=[N:1]CC[N:1](CC[N:1]=Cc1n(C)cc[n...,AMOHAM
4,4,csd_004,C(c1c[nH]c[n:1]1)=[N:1]CC[N:1](CC[N:1]=Cc1c[nH...,AMOJAO
...,...,...,...,...
104,104,csd_104,O=C(C[N:1](CC(=O)[OH:1])[C@H]1CCCC[C@@H]1[N:1]...,ZEBVEH
105,105,csd_105,c1ccc2c(c1)C1=N[C@]34N=c5c6ccccc6c([nH:1]5)=NC...,ZENGUU
106,106,csd_106,O=C(C[N:1]1CC[N:1](CC(=O)[OH:1])CC[N:1](Cc2ccc...,ZILDEH
107,107,csd_107,O=C(C[N:1]1CC[N:1](CC(=O)[OH:1])CC(=[O:1])NCC(...,ZIPJIR


In [7]:
df = pd.read_csv('../data/Gd/final/merged_reviews_clean.csv')
df

,ID,sourceID,smiles,doi,source,lgK
0,rev_081,81,CNC(=O)CN(CCN(CCN(CC(=O)O)CC(=O)NC)CC(=O)O)CC(...,10.1016/0730-725x(90)90055-7,dioury2014,16.85
1,rev_022,22,O=C(O)CN1CCN(CC(=O)O)CCN(CC(=O)O)CCN(CC(=O)O)CC1,10.1016/0020-1693(96)05182-1,dioury2014,24.67
2,rev_142,142,O=C(O)CN(CCN(CCN(CC(=O)O)CC(=O)O)CC(=O)O)CCN(C...,10.1016/0223-5234(88)90094-3,dioury2014,28.40
3,rev_035,35,CC(CO)N1CCN(CC(=O)O)CCN(CC(=O)O)CCN(CC(=O)O)CC1,10.1021/ic00095a028,dioury2014,23.90
4,rev_170,A39,O=C(O)CN(CC(=O)O)Cc1cccc(CN(CC(=O)O)CC(=O)O)n1,10.1039/b817343e,uzal2022,18.60
...,...,...,...,...,...,...
212,rev_136,136,CN(CC(O)C(O)C(O)C(O)CO)C(=O)C(COCc1ccccc1)N1CC...,10.1021/ic00038a023,dioury2014,26.40
213,rev_074,74,CCN(CCN(CC(=O)O)CC(=O)O)CCN(CC(=O)O)CC(=O)O,10.1021/ic00212a005,dioury2014,17.79
214,rev_227,T12,CCCCNC(=O)CN(CCN(CCN(CC(=O)O)CC(=O)O)CC(=O)O)C...,10.1039/C7DT04104G,uzal2022,18.78
215,rev_062,62,Nc1ccc(CC(C(=O)O)N(CCN(CC(=O)O)CC(=O)O)CCN(CC(...,10.1021/jm9602118,dioury2014,21.99


In [ ]:
csd_mols = []
csd_fps = []
csd_smiles_list = df_csd['smiles'].tolist()

for i, csd_smi in enumerate(csd_smiles_list):
    mol = None
    try:
        mol = Chem.MolFromSmiles(csd_smi)
    except:
        pass
    
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
        csd_mols.append(mol)
        csd_fps.append(fp)
    else:
        csd_mols.append(None)
        csd_fps.append(None)

data = []

for i, smiles_df1 in enumerate(df['smiles']):
    mol_df1 = None
    try:
        mol_df1 = Chem.MolFromSmiles(smiles_df1)
    except:
        pass

    if not mol_df1:
        data.append({
            'smiles': smiles_df1,
            'smiles_mol': None,
            'smiles_csd': None,
            'smiles_csd_mol': None,
            'sim': 0.0
        })
        continue

    fp_df1 = AllChem.GetMorganFingerprintAsBitVect(mol_df1, 2, nBits=2048)
    
    max_sim = -1
    best_csd_smiles = None
    best_csd_mol = None

    for j, fp_csd in enumerate(csd_fps):
        if fp_csd is not None:
            sim = FingerprintSimilarity(fp_df1, fp_csd)
            if sim > max_sim:
                max_sim = sim
                best_csd_smiles = csd_smiles_list[j]
                best_csd_mol = csd_mols[j]

    data.append({
        'smiles': smiles_df1,
        'smiles_mol': mol_df1,
        'smiles_csd': best_csd_smiles,
        'smiles_csd_mol': best_csd_mol,
        'sim': max_sim
    })

df_sim = pd.DataFrame(data)

PandasTools.AddMoleculeColumnToFrame(df_sim, 'smiles', 'smiles_img')
PandasTools.AddMoleculeColumnToFrame(df_sim, 'smiles_csd', 'smiles_csd_img')

PandasTools.SaveXlsxFromFrame(df_sim, '../data/Gd/df_similarity_results.xlsx',
                               molCol=['smiles_img', 'smiles_csd_img'])

[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerator
[16:35:35] DEPRECATION WARNING: please use MorganGenerat

In [ ]:
PandasTools.AddMoleculeColumnToFrame(df_csd, smilesCol='smiles', molCol='mol_image')
PandasTools.SaveXlsxFromFrame(df_csd, '../data/Gd/df_csd_with_images.xlsx', molCol='mol_image')